Stage 5 is about checking that it works reliably, identifying weaknesses, and determining whether the recommendations are reasonable.

In [38]:
#import packages and load data and features

import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("../data/processed/books_clean.csv")

combined_features = load_npz(
    "../data/processed/book_combined_features.npz"
)

similarity_matrix = cosine_similarity(combined_features)

In [39]:
#check dimensions

print("Number of books:", len(df))
print("Feature matrix shape:", combined_features.shape)
print("Similarity matrix shape:", similarity_matrix.shape)

Number of books: 99
Feature matrix shape: (99, 776)
Similarity matrix shape: (99, 99)


Rebuild the recommender. Try to do this from scratch!

In [40]:
def recommend_books_for_multiple(books_read, top_n=5): #we'll return 5 recommendations

    #make sure books are in dataset
    titles =[]
    indices = []

    for title in books_read:
        n_title = title.casefold()

        #skips the rest of the loop if the title is already stored
        if n_title in titles:
            continue

        matches = df.index[
            df["title"].str.casefold() == n_title
        ].tolist()

        #in case a user enters a book not in the dataset
        if not matches:
            raise ValueError(f"Book not found: {title}")

        indices.append(matches[0])
        titles.append(n_title)

    if not indices:
        raise ValueError("Please provide at least one book title.")

    average_scores = similarity_matrix[indices].mean(axis=0)
    scores = pd.Series(average_scores, index=df.index)

    # Exclude books the user has already read
    scores = scores[
        ~df["title"].str.casefold().isin(titles) #~ means not
    ]

    #ranks the remaining books and returns the most similar book
    top_indices = scores.sort_values(
        ascending=False
    ).head(top_n).index

    #retrieve book info
    results = df.loc[
        top_indices,
        ["book_id", "title", "author", "genres", "description", "average_rating"]
    ].copy() #creating a separate copy of data

    #appends similarity score to results table
    results["similarity_score"] = scores.loc[top_indices].values

    return results.reset_index(drop=True)


In [41]:
#testing normal request

recommendations = recommend_books_for_multiple(
    ["The Hobbit", "Dune"],
    top_n=5
)

recommendations

,book_id,title,author,genres,description,average_rating,similarity_score
0,B020,The Fellowship of the Ring,J. R. R. Tolkien,Fantasy;Epic;Adventure,A hobbit inherits a dangerous artifact and joi...,4.38,0.151835
1,B015,The Wise Man's Fear,Patrick Rothfuss,Fantasy;Magic;Adventure,The continuing account of a gifted young man's...,4.54,0.143131
2,B002,Hyperion,Dan Simmons,Science Fiction;Space Opera;Adventure,Seven travelers journey toward a mysterious wo...,4.22,0.122549
3,B003,The Left Hand of Darkness,Ursula K. Le Guin,Science Fiction;Politics;Literary,An envoy visits a distant planet and must navi...,4.08,0.115385
4,B009,Snow Crash,Neal Stephenson,Science Fiction;Cyberpunk;Adventure,A hacker and courier investigate a mysterious ...,4.00,0.112729


In [ ]:
#testing different n values

recommend_books_for_multiple(
    ["The Hobbit", "Dune"],
    top_n=3
)

recommend_books_for_multiple(
    ["The Hobbit", "Dune"],
    top_n=10
)

recommend_books_for_multiple(
    ["The Hobbit", "Dune"],
    top_n=0
)


,book_id,title,author,genres,description,average_rating,similarity_score
0,B020,The Fellowship of the Ring,J. R. R. Tolkien,Fantasy;Epic;Adventure,A hobbit inherits a dangerous artifact and joi...,4.38,0.151835
1,B015,The Wise Man's Fear,Patrick Rothfuss,Fantasy;Magic;Adventure,The continuing account of a gifted young man's...,4.54,0.143131
2,B002,Hyperion,Dan Simmons,Science Fiction;Space Opera;Adventure,Seven travelers journey toward a mysterious wo...,4.22,0.122549
3,B003,The Left Hand of Darkness,Ursula K. Le Guin,Science Fiction;Politics;Literary,An envoy visits a distant planet and must navi...,4.08,0.115385
4,B009,Snow Crash,Neal Stephenson,Science Fiction;Cyberpunk;Adventure,A hacker and courier investigate a mysterious ...,4.00,0.112729
5,B021,A Game of Thrones,George R. R. Martin,Fantasy;Epic;Politics,Competing noble families struggle for power wh...,4.44,0.112729
6,B006,The Martian,Andy Weir,Science Fiction;Survival;Adventure,An astronaut stranded on Mars uses engineering...,4.40,0.112729
7,B074,Into the Wild,Jon Krakauer,Biography;Adventure;Nature,A young man abandons conventional life and tra...,4.00,0.112729
8,B005,Foundation,Isaac Asimov,Science Fiction;Space Opera;Politics,A mathematician predicts the collapse of a vas...,4.17,0.111187
9,B007,Project Hail Mary,Andy Weir,Science Fiction;Space;Adventure,An astronaut awakens alone on a desperate miss...,4.50,0.108628


In [46]:
print(recommendations["similarity_score"].describe())

count    5.000000
mean     0.129126
std      0.017413
min      0.112729
25%      0.115385
50%      0.122549
75%      0.143131
max      0.151835
Name: similarity_score, dtype: float64


In [47]:
import numpy as np

scores = recommendations["similarity_score"]

print("All scores are numeric:", pd.api.types.is_numeric_dtype(scores))
print("All scores are finite:", np.isfinite(scores).all())

All scores are numeric: True
All scores are finite: True


In [48]:
print("Similarity matrix shape:", similarity_matrix.shape)

print(
    "First book's self-similarity:",
    similarity_matrix[0, 0]
)

Similarity matrix shape: (99, 99)
First book's self-similarity: 0.9999999999999998


In [50]:
#ensure recommendations are reproducible

first_run = recommend_books_for_multiple(
    ["The Hobbit", "Dune"],
    top_n=5
)

second_run = recommend_books_for_multiple(
    ["The Hobbit", "Dune"],
    top_n=5
)

first_run.equals(second_run)

True

We are going to test recommendation quality.

Look at the results. Do the recommendations appear to share meaningful features with the input books? Are they all from the same genre when they shouldn't be? Are descriptions dominating the results? Are there repeated authors or duplicate works?

In [ ]:
recommend_books_for_multiple(
    ["The Hobbit"],
    top_n=5
) #it's good the fantasy and adventure books are higher. They're all fantasy except the last one, but i've never read it
#so I can't for sure determine whether it's a good or bad rec

recommend_books_for_multiple(
    ["Dune"],
    top_n=5
) #they're all scifi so seems close enough? never read them so idk

recommend_books_for_multiple(
    ["Pride and Prejudice"],
    top_n=5
) #idk about the recommendations for this one. I think Jane Eyre should be higher than TGG. TKAM seems misplaced.

recommend_books_for_multiple(
    ["The Hobbit", "Dune", "Pride and Prejudice"],
    top_n=5
) #i think this is recommending based off the words "young man/woman" lol. desc

,book_id,title,author,genres,description,average_rating,similarity_score
0,B044,The Great Gatsby,F. Scott Fitzgerald,Classic;Literary;Romance,"A young man observes wealth, obsession, and so...",3.93,0.153676
1,B042,Jane Eyre,Charlotte Bronte,Romance;Gothic;Classic,An orphaned young woman becomes a governess an...,4.15,0.141962
2,B015,The Wise Man's Fear,Patrick Rothfuss,Fantasy;Magic;Adventure,The continuing account of a gifted young man's...,4.54,0.139965
3,B050,The Invisible Life of Addie LaRue,V. E. Schwab,Fantasy;Romance;Historical,A young woman makes a supernatural bargain for...,4.20,0.123702
4,B074,Into the Wild,Jon Krakauer,Biography;Adventure;Nature,A young man abandons conventional life and tra...,4.00,0.121378


Conclusions/Observations

- adding too many books muddies the waters a bit, possible due to the limited size of the testing dataset
- there were a lot of good recs for The Hobbit, not so much for Dune and Pride and Prejudice. Again, this could be a result of the limited dataset
- I think there's a good balance between description and genre. sometimes description is given too much weight, but with the small dataset
we don't really see the effects of this.

I think next steps will be to add more features to the model (such as rating) and then to expand the dataset.